# 01. Dựng dataset

Sinh KB tổng hợp tiếng Việt (2 tenant), xuất mẫu kiểm tra nhãn, tải ViNLI. Phần Jira là tuỳ chọn.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "https://github.com/vinhlee9x/conflict-detection-obsolescence-forecasting.git", "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not os.path.exists("src") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Thư mục làm việc:", os.getcwd())

In [ ]:
# Tuỳ chọn: giữ data/ và artifacts/ trên Google Drive giữa các session Colab
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/conflict-detection"
    for folder in ("data/processed", "artifacts"):
        os.makedirs(f"{target}/{folder}", exist_ok=True)
        subprocess.run(f"cp -rn {folder}/. {target}/{folder}/ && rm -rf {folder} && ln -s {target}/{folder} {folder}", shell=True, check=True)

## KB tổng hợp

In [ ]:
from src import pipeline
from src.io_utils import read_parquet
from src.paths import DATA_PROCESSED
pipeline.generate_kb()
docs = read_parquet(DATA_PROCESSED / "kb_docs.parquet")
pairs = read_parquet(DATA_PROCESSED / "kb_pairs.parquet")
edges = read_parquet(DATA_PROCESSED / "kb_edges.parquet")
print(len(docs), "tài liệu,", len(pairs), "cặp,", int(pairs.label.sum()), "cặp mâu thuẫn")
pairs.groupby(["tenant", "variant_type", "label"]).size().unstack("tenant")

In [ ]:
example = pairs[pairs.variant_type == "conflict"].iloc[0]
text = docs.set_index("doc_id")["text"]
print(text[example.doc_a]); print("-" * 60); print(text[example.doc_b])

## Mẫu 100 cặp để nhóm gán nhãn lại

Mỗi người thêm một cột `annot_<tên>` (1 = mâu thuẫn, 0 = không), rồi chạy ô tính kappa.

In [ ]:
from src.synth.qa_sample import export_blind_sample
sample = export_blind_sample(docs, pairs, n=100, seed=42)
sample.to_csv(DATA_PROCESSED / "kb_qa_sample.csv", index=False)
sample.head()

In [ ]:
from pathlib import Path
import pandas as pd
from src.synth.qa_sample import kappa_report, disagreements
annotated_path = DATA_PROCESSED / "kb_qa_sample_annotated.csv"
if annotated_path.exists():
    annotated = pd.read_csv(annotated_path)
    display(kappa_report(annotated, pairs))
    display(disagreements(annotated, pairs))
else:
    print("Chưa có file gán nhãn:", annotated_path)

## ViNLI

In [ ]:
from src.datasets.vinli import load_vinli
vinli_test = load_vinli("test")
vinli_test.label.value_counts()

## Jira (tuỳ chọn, giới hạn 6 giờ)

Tải dump từ Zenodo (doi 10.5281/zenodo.5882881), cài MongoDB, `mongorestore`, rồi đặt `RUN_JIRA = True`.

In [ ]:
RUN_JIRA = False
if RUN_JIRA:
    from src.datasets.jira_export import export_project
    from src.datasets.jira_link_stats import link_type_counts, resolution_counts, project_summary
    from src.io_utils import write_parquet
    jira_docs, jira_edges = export_project("mongodb://localhost:27017", "JiraRepos", "Apache", project_key="CASSANDRA")
    write_parquet(jira_docs, DATA_PROCESSED / "jira_docs.parquet")
    write_parquet(jira_edges, DATA_PROCESSED / "jira_edges.parquet")
    display(project_summary(jira_docs, jira_edges))
    display(link_type_counts(jira_edges).head(20))
    display(resolution_counts(jira_docs))

In [ ]:
if RUN_JIRA:
    from src.datasets.jira_pairs import build_pairs
    pipeline.embed("jira", backend="bge-m3")
    neighbors = read_parquet(pipeline.artifact_dir("jira") / "neighbors.parquet")
    jira_pairs = build_pairs(jira_docs, jira_edges, neighbors, seed=42)
    write_parquet(jira_pairs, DATA_PROCESSED / "jira_pairs.parquet")
    jira_pairs.groupby(["variant_type", "label"]).size()